# Gene sets round trip: GMT in, signature scores out

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/python_package/f_notebooks_tutorials/45_gene_sets_round_trip.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Gene sets travel both ways. Python sends a set to the viewer
(`viewer.push_gene_set`), where **Gene signature** and **Dot plot**
offer it under **From notebook**; the viewer's sets (yours, its marker
genes, a signature's) come back with `viewer.gene_sets()`, and the
scores a gene signature computes come back as `obs` columns with
`viewer.pull`. GMT files carry sets between Python, R and the web app.

**Time:** 10 minutes. The same tutorial in R:
[Gene sets round trip in R](../../r_package/f_notebooks_tutorials/11_gene_sets_round_trip.ipynb).

In [ ]:
# A fresh runtime (Google Colab, a new environment) installs Cellucid once.
import importlib.util

if importlib.util.find_spec("cellucid") is None:
    %pip install --quiet cellucid

## The data

The 2,531-cell mouse pancreas (endocrinogenesis, day 15.5) AnnData checked
into the cellucid-python repository: UMAP in `obsm['X_umap_2d']`, cell types
in `obs['clusters']`, 27,998 genes. In a clone of the repository the cell
below finds it; anywhere else (Colab) it downloads it once. Use your own
`adata` in its place.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import anndata as ad

import cellucid

NAME = "endocrinogenesis_day15.5_raw.h5ad"
URL = (
    "https://raw.githubusercontent.com/theislab/cellucid-python/main/"
    "docs/user_guide/python_package/f_notebooks_tutorials/datasets/" + NAME
)


def pancreas_h5ad() -> Path:
    """The Pancreas file of a cellucid-python clone, else a copy downloaded once (35 MB)."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for candidate in (base / "datasets" / NAME, base / "docs/user_guide/python_package/f_notebooks_tutorials/datasets" / NAME):
            if candidate.is_file():
                return candidate
    if not (here / NAME).is_file():
        urlretrieve(URL, here / NAME)
    return here / NAME


DATA_PATH = pancreas_h5ad()
adata = ad.read_h5ad(DATA_PATH)
adata

## Gene sets as a GMT file

A GMT file (MSigDB's format) holds one set per line:
`name<TAB>description<TAB>gene…`. `write_gmt` and `read_gmt` write and
read it; the web app's gene set editor imports and exports the same
format.

In [ ]:
markers = {
    "Beta": ["Ins1", "Ins2", "Iapp", "Pdx1", "Nkx6-1"],
    "Alpha": ["Gcg", "Arx", "Irx2"],
    "Delta": ["Sst", "Hhex"],
    "Epsilon": ["Ghrl"],
}
path = cellucid.write_gmt(markers, "pancreas_markers.gmt", descriptions={"Beta": "insulin-producing"})
print(path.read_text())
assert cellucid.read_gmt(path) == markers

## Into the viewer

Each set is sent at once; a set of the same name replaces the one
sent before, and the sets are saved with the session.

In [ ]:
DATASET_ID = "pancreas-e15-5"

viewer = cellucid.show_anndata(DATA_PATH, dataset_name="Pancreas E15.5", dataset_id=DATASET_ID)

In [ ]:
for name, genes in markers.items():
    viewer.push_gene_set(name, genes)

## Score them in the viewer

In **Analysis › Gene signature**, choose **From notebook** and pick
`Beta`; run it. Do the same for `Alpha`. Optionally, run **Marker
genes** on `clusters` and keep its result shown. Then run the next
cell.

In [ ]:
summary = viewer.pull(adata)
scores = [name for name, info in summary.columns.items() if info.get("origin") == "analysis"]
print("scores:", scores)
adata.obs.groupby("clusters", observed=True)[scores].mean().round(2)

Each score is a `float32` column named after its set
(`<set> score`). `viewer.gene_sets()` lists every set the viewer
holds: those you pushed, a marker genes result's (one per group) and a
signature's.

In [ ]:
held = viewer.gene_sets()
print(list(held))
cellucid.write_gmt(held, "from_viewer.gmt")

`pull` also kept them in `adata.uns['cellucid']['gene_sets']`, each
with its genes, description and origin: `notebook` (pushed),
`markers` (a marker genes result) or `analysis` (a gene signature's).

In [ ]:
{name: entry["origin"] for name, entry in adata.uns["cellucid"]["gene_sets"].items()}

In [ ]:
viewer.stop()

## Next

- [Export a subset and re-analyse it](46_export_an_h5ad_subset_and_reanalyse.ipynb)
- R: [Gene sets round trip in R](../../r_package/f_notebooks_tutorials/11_gene_sets_round_trip.ipynb)